# Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.utils import iers
iers.conf.auto_download = False          # brez prenosa tabel z interneta
import astropy.units as u
from astropy.coordinates import EarthLocation, AltAz, get_sun
from astropy.time import Time

NAVODILA = Path("navodila") if Path("navodila").exists() else Path("../navodila")

# Lokacija: FNM, Koroška cesta 160, Maribor
phi, lam = 46.559, 15.639       # geografska širina in dolžina [°]
S0 = 1361.0                     # solarna konstanta [W/m^2]
leto = 2023
d2r = np.deg2rad

# Funkcije

In [ ]:
def dan_v_letu(datum):
    return pd.Timestamp(datum).dayofyear


def deklinacija(n):
    """Deklinacija Sonca [°] (Cooperjeva enačba)."""
    # TODO
    pass


def casovna_enacba(n):
    """Časovna enačba [min]: razlika med pravim in srednjim sončnim časom."""
    B = 2 * np.pi * (n - 81) / 364
    return 9.87 * np.sin(2 * B) - 7.53 * np.cos(B) - 1.5 * np.sin(B)


def urni_kot(t_utc, n, lam=lam):
    """Urni kot [°] za čas t_utc [h]; 0° ob pravem sončnem poldnevu, pozitiven popoldne."""
    # TODO: pravi sončni čas t_s = t_utc + lam/15 + E/60
    pass


def visina(phi, delta, H):
    """Višina Sonca nad obzorjem [°]."""
    # TODO
    pass


def azimut(phi, delta, H):
    """Azimut Sonca [°], merjen od severa v smeri urinega kazalca."""
    alfa = d2r(visina(phi, delta, H))
    # TODO: cos A = (sin delta - sin alfa sin phi) / (cos alfa cos phi); popoldne A -> 360 - A
    pass


def dolzina_dneva(phi, delta):
    """Dolžina dneva [h] iz cos H0 = -tan(phi) tan(delta)."""
    # TODO
    pass


def vzhod_zahod(n, phi=phi, lam=lam):
    """Čas sončnega vzhoda in zahoda [h, UTC]."""
    H0 = dolzina_dneva(phi, deklinacija(n)) * 15 / 2
    poldan = 12 - lam / 15 - casovna_enacba(n) / 60
    return poldan - H0 / 15, poldan + H0 / 15


def zracna_masa(alt):
    """Relativna zračna masa (Kasten in Young, 1989); alt v stopinjah."""
    # TODO
    pass


def obsevanje_jasno(alt, n):
    """Globalno obsevanje vodoravne ploskve v jasnem vremenu [W/m^2].
    Direktno: I_D = S0' 0.7^(AM^0.678), globalno ~ 1.1 I_D, na vodoravno ploskev: * sin(alt)."""
    S0n = S0 * (1 + 0.033 * np.cos(2 * np.pi * n / 365))
    # TODO: 0 za alt <= 0
    pass


def astropy_lega(datum, korak_min=10):
    """Višina in azimut Sonca iz knjižnice astropy za cel dan (UTC)."""
    loc = EarthLocation(lat=phi * u.deg, lon=lam * u.deg, height=270 * u.m)
    t = Time(f"{datum}T00:00:00") + np.arange(0, 24 * 60, korak_min) * u.min
    aa = get_sun(t).transform_to(AltAz(obstime=t, location=loc))
    return np.arange(0, 24 * 60, korak_min) / 60, aa.alt.deg, aa.az.deg

# Testno območje

In [ ]:
for datum in ("2023-03-21", "2023-06-21", "2023-12-21"):
    n = dan_v_letu(datum)
    d = deklinacija(n)
    print(f"{datum}: n = {n}, delta = {d:6.2f}°, alt_max = {90 - phi + d:5.2f}°, "
          f"dan = {dolzina_dneva(phi, d):5.2f} h, E = {casovna_enacba(n):5.1f} min")
print("zračna masa pri 90°, 30°, 10°, 0°:", np.round(zracna_masa([90, 30, 10, 0]), 2))

# Naloga 1 - Slika 2

In [ ]:
dnevno = pd.read_csv(NAVODILA / "solar_daily.csv", index_col=0)
n_v = np.arange(1, 366)
# TODO: dolžina dneva in največja višina (model) v primerjavi s podatki (astropy); Preglednica 1
dan_model = alt_model = None
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121)
# TODO
fig.tight_layout()
plt.show()
for n in (1, 80, 172, 265, 355):
    r = dnevno.iloc[n - 1]
    print(f"n = {n:3d}: dan {dan_model[n-1]:5.2f} / {r['Dolžina dneva']:5.2f} h, "
          f"alt {alt_model[n-1]:5.2f} / {r['Alt maks']:5.2f}°, vzhod {vzhod_zahod(n)[0]:5.2f} / {r['Sončni vzhod ura']:5.2f} h UTC")

# Naloga 2 - Slika 3

In [ ]:
datumi = ["2023-03-21", "2023-06-21", "2023-12-21"]
t = np.arange(0, 24, 1 / 6)
fig = plt.figure(figsize=(12, 4.5))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection="polar")
# TODO: višina(t) za tri dni in pot Sonca v polarnem grafu (azimut, 90 - višina)
fig.tight_layout()
plt.show()

# Naloga 3 - Slika 4

In [ ]:
fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
# TODO: primerjava modela z astropy (višina in azimut) za 21. 6.; največje razlike (Preglednica 3)
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 5

In [ ]:
vreme = pd.read_csv(NAVODILA / "Vreme_urni.csv")
vreme["cas"] = pd.to_datetime(vreme["date"] + " " + vreme["time"])
tt = np.arange(0, 24, 1 / 6)                       # vsakih 10 minut
# TODO: dnevna energija v jasnem vremenu [kWh/m^2] za vse dni; izmerjena dnevna energija (vsota J * 1 h); indeks jasnosti po mesecih
E_jasno = None
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(111)
# TODO
plt.show()
mesec = pd.to_datetime(pd.Series(n_v), unit="D", origin=f"{leto - 1}-12-31").dt.month.to_numpy()
for m in (1, 3, 6, 9, 12):        # Preglednica 4
    print(f"mesec {m:2d}: jasno = {E_jasno[mesec == m].mean():.2f}, izmerjeno = {E_mer[mesec == m].mean():.2f} kWh/m^2/dan, "
          f"indeks jasnosti = {E_mer[mesec == m].mean() / E_jasno[mesec == m].mean():.2f}")
print(f"letno: jasno = {E_jasno.sum():.0f}, izmerjeno = {E_mer.sum():.0f} kWh/m^2")